In [1]:
import scarf

scarf.set_verbosity('WARNING')

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    'tenx_5K_pbmc_rnaseq',
    destination='scarf_datasets',
    zarr=True,
)
ds = scarf.DataStore(
    f'{dataset}/data.zarr',
    nthreads=4,
    min_features_per_cell=10,
)
ds.filter_cells(
    attrs=['RNA_nCounts', 'RNA_nFeatures'],
    highs=[15000, 4000],
    lows=[1000, 500],
    reset_previous=True,
)
if 'I__hvgs' not in ds.RNA.feats.columns:
    ds.mark_hvgs(min_cells=20, top_n=500, show_plot=False)

In [2]:
normalized = ds.run_normalization(feat_key='hvgs')
pca = ds.run_pca(normalized, dims=15)
init = ds.build_embedding_initialization(pca, n_centroids=100)
ann = ds.build_ann_index(pca)
neighbors = ds.query_neighbors(ann, k=11)
graph = ds.build_connectivity_map(neighbors)

state = ds.get_assay_state('RNA')
(
    state.normalized,
    state.reduction,
    state.embedding_initialization,
    state.connectivity_map,
)

(ArtifactRef(assay='RNA', kind='normalized', artifact_id='fa7780238622...'),
 ArtifactRef(assay='RNA', kind='reduction', artifact_id='cac4a065079c...'),
 ArtifactRef(assay='RNA', kind='embedding_initialization', artifact_id='429c6e9e5e6a...'),
 ArtifactRef(assay='RNA', kind='connectivity_map', artifact_id='b231fac806f2...'))

In [3]:
ds.run_umap(n_epochs=100, parallel=True)
ds.run_leiden_clustering(resolution=0.5)